# AP CSP Supplemental Practice Exam — Full Performance Report

**Date:** May 8, 2026
**Student:** Akhil
**Exams:** Supplemental Practice Exam MCQ (70 questions) + Written Response Supplemental Practice Exam (FRQ)

This blog walks through my full results on the AP CSP Supplemental Practice Exam — both the multiple-choice section and the written-response (FRQ) Create Task. The goal is to use this as a focused review document leading into the AP exam: identify exactly which question types tripped me up, lock in the AP CSP concepts behind those misses, and rehearse the FRQ rubric on the StarMatch project I built earlier.

## Section 1 — MCQ Performance Report

### Score Summary

| Metric | Result |
|--------|--------|
| **Total Questions** | 70 |
| **Correct** | ~65 |
| **Incorrect** | ~5 |
| **Score** | **~92.9%** |

### Questions / Topics That Were Challenging

- **Q5 — Code Maintainability / Abstraction:** Initially selected "putting each duplicated segment into a separate procedure called once" — wrong because that still duplicates logic. The correct answer is putting all duplicated code into ONE procedure called multiple times. *Takeaway: abstraction means one function, not many.*

- **Q12 — Flowchart Logic (Missing Step):** Needed careful tracing of the flowchart to determine the condition was `ounces > 10` (not `ounces ≥ 10` or any OR condition). The structure of nested IF in flowcharts requires paying close attention to which branch leads where.

- **Q28 — Token Display Procedure Bug:** The third REPEAT UNTIL loop (`REPEAT UNTIL(total = 0)`) is missing a `total ← total − 1` decrement. For `displayTokens(19)`, after distributing TENs and FIVEs, remaining total = 4 — but the loop has no decrement, so it would never terminate for non-zero remainders. The other calls (80, 35, 5) happen to leave total = 0 going into that loop, so they appear to work.

- **Q27 & Q40 — Robot Grid Navigation:** These require methodically tracing each procedure (skipLeft, skipRight, MOVE_FORWARD with rotation) step by step. Error-prone because it's easy to lose track of robot orientation after each rotation.

- **Q43 — Fault-Tolerance Network Graph:** My first instinct selected B (B and E fail → A can't reach F). This was wrong — traced the path A→D→C→F which still works. Correct answer is A (B and C fail → F becomes isolated since F only connects to C).

### How I Intend to Improve

- **Abstraction/Procedures:** Practice identifying the difference between "multiple procedures called once each" (no benefit) vs. "one procedure called multiple times" (true abstraction). This is a core AP CSP concept tested repeatedly.

- **Flowchart Tracing:** Always label which side is "true" and which is "false" before reading the conditions. Sketch it out.

- **Loop Bugs with REPEAT UNTIL:** Remember that REPEAT UNTIL in AP pseudocode checks condition *after* executing the body (do-while style). Missing a decrement inside a loop is a classic bug — look for it explicitly.

- **Robot Grid Problems:** Use a physical grid sketch. Mark the robot's position AND direction at each step. Don't rely on mental simulation.

- **Network Graph Connectivity:** For "which failures disconnect X from Y," systematically enumerate all paths from X to Y, then check which path-breaking combinations are covered by each answer choice.

## Section 2 — FRQ Written Response: StarMatch

### Self-Scoring (CollegeBoard Rubric)

The AP CSP Create Task is scored on 6 points. My self-assessment:

| Row | Criteria | Score | Reasoning |
|-----|----------|-------|-----------|
| **Row 1** | Program Purpose & Function | **1/1** | Video (PPR) would show input, program running, output; written response clearly explains inputs and their effect |
| **Row 2** | Data Abstraction | **1/1** | `celestial_objects` is a list of dictionaries managing complexity; code shows data stored in and used from this list |
| **Row 3** | Managing Complexity | **1/1** | The list enables the scoring loop to process any number of objects without separate variables for each |
| **Row 4** | Procedural Abstraction | **1/1** | `calculate_viewing_score` is a student-developed procedure with parameters that affects functionality |
| **Row 5** | Algorithm Implementation | **1/1** | Procedure contains sequencing, selection (IF/ELIF/ELSE), and iteration (for-loop over best_months) |
| **Row 6** | Testing | **1/1** | Two calls produce same result with different arguments; describes the path leading to same output |
| **TOTAL** | | **6/6** | Full score |

### FRQ Prompt Explanations (Part by Part)

**Question 1 — Program Input/Output:**

- *Prompt:* Identify an input your program receives and explain how it affects what the program does.
- *My Response:* StarMatch accepts four inputs — month (1–12), experience (1–5), minutes available, and pollution level (1–3). Each input is passed to `calculate_viewing_score()` and directly modifies the integer score assigned to each celestial object. Month determines the ±season bonus/penalty; experience level is compared to each object's difficulty for a bonus/penalty; pollution penalizes dim objects in urban skies; minutes penalizes hard objects when time is short. The top 5 highest-scoring objects are then output.

**Question 2a — Iteration:**

- *Prompt:* What values would cause the first iteration statement's body to execute zero times?
- *My Response:* The first loop is `for m in obj["best_months"]`. Because every object's `best_months` list is initialized with 3–6 values and is never empty, there are **no values** that cause zero executions. The loop body always runs at least once.

**Question 2b — Two Calls, Same Result:**

- *Prompt:* Write two different calls that produce the same result.
- *My Response:*
  - Call 1: `calculate_viewing_score(jupiter_obj, 5, 1, 60, 2)` → score = 45
  - Call 2: `calculate_viewing_score(pleiades_obj, 3, 1, 60, 2)` → score = 45
  - Both: month is out-of-season (−20), difficulty matches experience (+ 15), suburban pollution (no adjustment), plenty of time (no penalty). Score = 50 − 20 + 15 = **45**.

**Question 2c — Advantage of the Procedure:**

- *Prompt:* How would you adjust the program if the procedure didn't exist? What's one advantage of using it?
- *My Response:* Without the procedure, all scoring logic would be inlined inside `main()`'s loop, making it much longer. The key advantage is **modularity/abstraction**: if scoring rules change, only one function needs updating rather than scattered inline code.

### Code Deep Dive — StarMatch Explained

The StarMatch project is broken into a data layer (`celestial_objects`), a scoring procedure (`calculate_viewing_score`), an input layer (`get_user_input`), and a `main()` driver. The next few cells walk through each piece in the order it appears in the program.

#### Data Structure — list of dictionaries

In [ ]:
# ── DATA STRUCTURE ──────────────────────────────────────────────────────────
celestial_objects = [
    {"name": "Orion Nebula", "type": "Nebula", "best_months": [11,12,1,2,3],
     "difficulty": 2, "magnitude": 4.0,
     "fact": "A stellar nursery where new stars are forming right now."},
    # ... 11 more objects
]

> **Why a list of dictionaries?**
>
> Each dictionary is a structured record for one sky object, storing 5 attributes. The outer list lets the program iterate over all 12 objects with a single `for` loop — this is the **list managing complexity** required by CollegeBoard Row 2/3.

#### Scoring Procedure — `calculate_viewing_score`

In [ ]:
def calculate_viewing_score(obj, current_month, experience,
                            minutes_available, pollution_level):
    score = 50  # baseline

    # ── ITERATION: search best_months list ──────────────────────────────────
    month_match = False
    for m in obj["best_months"]:          # ← iteration over list
        if m == current_month:            # ← selection
            month_match = True

    # ── SELECTION: season bonus/penalty ─────────────────────────────────────
    if month_match:
        score += 30
    else:
        score -= 20

    # ── SELECTION: experience vs. difficulty ─────────────────────────────────
    if obj["difficulty"] <= experience:
        score += 15
    elif obj["difficulty"] == experience + 1:
        score -= 5
    else:
        score -= 25

    # ── SELECTION: light pollution ───────────────────────────────────────────
    if pollution_level == 1:
        score += 10
    elif pollution_level == 3:
        if obj["magnitude"] > 3:
            score -= 30
        else:
            score -= 5

    # ── SELECTION: time availability ─────────────────────────────────────────
    if obj["difficulty"] >= 4 and minutes_available < 30:
        score -= 20

    if score < 0:
        score = 0    # clamp to minimum 0

    return score

**Three required algorithm components present:**

1. **Sequencing** — statements execute top to bottom in fixed order (season → experience → pollution → time → clamp)
2. **Selection** — every `if/elif/else` block
3. **Iteration** — `for m in obj["best_months"]` loop

#### Driver — `main()`

In [ ]:
def main():
    month, experience, minutes, pollution = get_user_input()

    # Score every object
    scored_objects = []
    for obj in celestial_objects:                           # iteration
        score = calculate_viewing_score(                    # procedure call
            obj, month, experience, minutes, pollution)
        scored_objects.append((obj, score))                 # list append

    # Sort highest-score first
    scored_objects.sort(key=lambda pair: pair[1], reverse=True)

    # Display top 5
    top_n = min(5, len(scored_objects))
    for i in range(top_n):
        obj, score = scored_objects[i]
        print(f"{i+1}. {obj['name']}  ({obj['type']})")
        print(f"   Match score: {score}/100")
        print(f"   Did you know: {obj['fact']}")

**This section shows:**

- The procedure being **called** (Row 4 PPR segment ii)
- The list **being used** — `scored_objects.append()` and `scored_objects[i]` (Row 3 PPR segment ii)
- Output based on input (Row 1)

### Possible CollegeBoard Questions & Answers About This Code

**Q: What is the purpose of the `celestial_objects` list?**

A: It stores all 12 celestial objects as structured dictionaries, allowing the program to manage them as a collection. Without it, 12 separate variables would be needed and the `for` loop couldn't generalize over all objects. This is the data abstraction that manages program complexity.

**Q: What does `calculate_viewing_score` return, and how is it used?**

A: It returns an integer (0–100). In `main()`, the return value is stored in `score` and paired with the object as a tuple in `scored_objects`. The list is then sorted by score to produce the ranked recommendation.

**Q: Identify the algorithm in the procedure and name its three components.**

A: The algorithm is the multi-factor scoring system. *Sequencing:* each scoring block executes in order. *Selection:* `if/elif/else` chains choose which bonus/penalty to apply. *Iteration:* the `for m in obj["best_months"]` loop checks each month in the list.

**Q: How does changing `experience` from 1 to 5 affect the output?**

A: A higher experience value means the condition `obj["difficulty"] <= experience` is true for more objects (including higher-difficulty ones like Ring Nebula difficulty=4 or Whirlpool Galaxy difficulty=5), so they gain +15 instead of −25. Harder objects move higher in the ranking.

**Q: Could the program produce incorrect results for a specific input? Give an example.**

A: If `pollution_level = 3` and an object has `magnitude ≤ 3` (e.g., Jupiter, magnitude = −2.0), the penalty is only −5. But the Ring Nebula (magnitude = 8.8) would be penalized −30. This is intentional: bright objects like planets are visible even in cities; dim nebulae are not.

**Q: What would happen if `scored_objects` were not sorted?**

A: The top-5 output would display the first 5 objects from the list in original insertion order rather than by score, defeating the purpose of the recommendation engine.

**Q: Trace the score for Jupiter (difficulty=1, best_months=[1,2,3,10,11,12]) with inputs month=5, experience=1, minutes=60, pollution=2.**

A: Start=50. Month 5 not in best_months → −20 → 30. Difficulty 1 ≤ experience 1 → +15 → 45. Pollution=2 → no change → 45. Minutes=60 ≥ 30 → no penalty → **final score = 45**.

**Q: How does the program handle the case where score goes negative?**

A: The line `if score < 0: score = 0` clamps the value. This prevents negative scores from distorting the sort order and ensures all recommendations are presented on a 0–100 scale.

## Full FRQ Code with Additional Comments

The complete StarMatch program below is the version annotated for review — it includes inline comments tying each section back to a specific CollegeBoard rubric row. (The actual PPR submission would have these comments stripped from the screenshots.)

In [ ]:
# ============================================================
# StarMatch — Personalized Stargazing Planner
# AP CSP Create Task Portfolio Project
# ============================================================
# PURPOSE: Ranks 12 celestial objects by a personalized
# "viewing worthiness" score (0-100) based on four user
# inputs: current month, observer experience, available
# time, and sky-quality/light-pollution level.
# ============================================================

# ── DATA ABSTRACTION (CollegeBoard Row 2 & 3) ──────────────
# A list of dictionaries manages all 12 objects as one
# collection. Iterating over this list in main() avoids
# the need for 12 separate variables and allows the scoring
# algorithm to be applied uniformly to every object.
celestial_objects = [
    {"name": "Orion Nebula",     "type": "Nebula",
     "best_months": [11,12,1,2,3], "difficulty": 2,
     "magnitude": 4.0,
     "fact": "A stellar nursery where new stars are forming right now."},

    {"name": "Andromeda Galaxy", "type": "Galaxy",
     "best_months": [9,10,11,12],  "difficulty": 3,
     "magnitude": 3.4,
     "fact": "On a collision course with the Milky Way in 4.5 billion years."},

    {"name": "Jupiter",          "type": "Planet",
     "best_months": [1,2,3,10,11,12], "difficulty": 1,
     "magnitude": -2.0,
     "fact": "Binoculars reveal its four largest moons, first seen by Galileo in 1610."},

    {"name": "Saturn",           "type": "Planet",
     "best_months": [6,7,8,9],   "difficulty": 1,
     "magnitude": 0.5,
     "fact": "Its rings are visible through almost any small telescope."},

    {"name": "Pleiades Cluster", "type": "Star Cluster",
     "best_months": [10,11,12,1,2], "difficulty": 1,
     "magnitude": 1.6,
     "fact": "Also called the Seven Sisters, mentioned in ancient texts worldwide."},

    {"name": "Ring Nebula",      "type": "Nebula",
     "best_months": [6,7,8,9],   "difficulty": 4,
     "magnitude": 8.8,
     "fact": "The expanding shell of a dying star, shaped like a smoke ring."},

    {"name": "Mars",             "type": "Planet",
     "best_months": [11,12,1],   "difficulty": 2,
     "magnitude": -0.5,
     "fact": "Polar ice caps are visible during opposition events."},

    {"name": "Whirlpool Galaxy", "type": "Galaxy",
     "best_months": [3,4,5,6],   "difficulty": 5,
     "magnitude": 8.4,
     "fact": "Two galaxies caught mid-collision in a gravitational dance."},

    {"name": "Hercules Cluster", "type": "Star Cluster",
     "best_months": [5,6,7,8],   "difficulty": 3,
     "magnitude": 5.8,
     "fact": "A globular cluster packed with hundreds of thousands of ancient stars."},

    {"name": "Venus",            "type": "Planet",
     "best_months": [3,4,5,9,10,11], "difficulty": 1,
     "magnitude": -4.0,
     "fact": "Goes through phases like the Moon when viewed through a telescope."},

    {"name": "Double Cluster",   "type": "Star Cluster",
     "best_months": [10,11,12,1], "difficulty": 2,
     "magnitude": 3.7,
     "fact": "Two open star clusters that look gorgeous side by side in binoculars."},

    {"name": "Lagoon Nebula",    "type": "Nebula",
     "best_months": [6,7,8],     "difficulty": 3,
     "magnitude": 6.0,
     "fact": "A massive star-forming region in the heart of Sagittarius."},
]


# ── STUDENT-DEVELOPED PROCEDURE (CollegeBoard Row 4 & 5) ───
# calculate_viewing_score takes 5 parameters, all of which
# have a meaningful effect on the return value.
# It contains SEQUENCING (steps in order), SELECTION
# (if/elif/else chains), and ITERATION (for-loop).
def calculate_viewing_score(obj, current_month, experience,
                            minutes_available, pollution_level):
    """
    Returns an integer score 0–100 representing how suitable
    'obj' is to observe tonight given the user's conditions.
    """
    score = 50  # neutral baseline before bonuses/penalties

    # ── ITERATION ──────────────────────────────────────────
    # Loop through every month in the object's best_months
    # list. If any element equals current_month, flag it.
    # Because every object has ≥ 3 months, this loop body
    # ALWAYS executes at least once (key FRQ fact).
    month_match = False
    for m in obj["best_months"]:      # ITERATION over list
        if m == current_month:        # SELECTION inside loop
            month_match = True

    # ── SELECTION: seasonality ─────────────────────────────
    # In-season objects get a large boost; out-of-season
    # objects receive a significant penalty.
    if month_match:
        score = score + 30    # in-season bonus
    else:
        score = score - 20    # out-of-season penalty

    # ── SELECTION: experience vs. difficulty ───────────────
    # Objects at or below the user's skill level are rewarded.
    # Objects one step above are slightly penalized (they're
    # achievable but challenging). Much harder objects are
    # penalized heavily.
    if obj["difficulty"] <= experience:
        score = score + 15    # within skill range
    elif obj["difficulty"] == experience + 1:
        score = score - 5     # slightly above skill
    else:
        score = score - 25    # far above skill level

    # ── SELECTION: light pollution ─────────────────────────
    # Dark skies (level 1) give a bonus for all objects.
    # Urban skies (level 3) only hurt dim objects (mag > 3);
    # bright planets/stars remain visible regardless.
    if pollution_level == 1:
        score = score + 10    # dark sky bonus
    elif pollution_level == 3:
        if obj["magnitude"] > 3:
            score = score - 30  # dim object in city: harsh
        else:
            score = score - 5   # bright object in city: mild

    # ── SELECTION: time constraint ─────────────────────────
    # High-difficulty objects (4+) need ≥ 30 min to observe
    # properly. Short sessions penalize them.
    if obj["difficulty"] >= 4 and minutes_available < 30:
        score = score - 20

    # ── CLAMPING ───────────────────────────────────────────
    # Prevent negative scores; the scale is always 0–100.
    if score < 0:
        score = 0

    return score   # integer return value used by main()


# ── INPUT FUNCTION ─────────────────────────────────────────
def get_user_input():
    """Collects the four user inputs from the console."""
    print("=" * 52)
    print("       STARMATCH - Tonight's Best Sky Targets")
    print("=" * 52)
    print()
    month      = int(input("Current month (1-12): "))
    experience = int(input("Your experience (1=beginner, 5=expert): "))
    minutes    = int(input("Minutes available to observe: "))
    pollution  = int(input("Sky conditions (1=dark, 2=suburban, 3=urban): "))
    return month, experience, minutes, pollution


# ── MAIN FUNCTION ──────────────────────────────────────────
def main():
    # Receive all four inputs
    month, experience, minutes, pollution = get_user_input()

    # Score every object; store (object, score) tuples in a list
    scored_objects = []                  # list starts empty
    for obj in celestial_objects:        # ITERATION over data list
        score = calculate_viewing_score( # procedure CALL
            obj, month, experience, minutes, pollution)
        scored_objects.append((obj, score))  # store result in list

    # Sort descending by score (lambda extracts the score element)
    scored_objects.sort(key=lambda pair: pair[1], reverse=True)

    # Output the top 5 personalized recommendations
    print()
    print("-" * 52)
    print("         YOUR PERSONALIZED VIEWING PLAN")
    print("-" * 52)
    top_n = 5
    if len(scored_objects) < top_n:
        top_n = len(scored_objects)

    for i in range(top_n):
        obj, score = scored_objects[i]
        print()
        print(str(i + 1) + ". " + obj["name"] + "  (" + obj["type"] + ")")
        print("   Match score: " + str(score) + "/100")
        print("   Did you know: " + obj["fact"])

    print()
    print("-" * 52)
    print("Clear skies!")

main()

## Exam-Day Strategy: How to Full-Score the Actual FRQ

Based on this code, here is my precise plan for the actual AP Exam FRQ:

- **PPR Procedure Segment 1 — the procedure itself:**
  Screenshot `calculate_viewing_score` from `def calculate_viewing_score(...)` through `return score`. This shows the procedure name, 5 parameters, the `for` loop (iteration), and multiple `if/elif/else` chains (selection). No comments allowed in the PPR — remove them for the actual submission screenshot.

- **PPR Procedure Segment 2 — the call site:**
  Screenshot the block inside `main()` that calls `calculate_viewing_score(obj, month, experience, minutes, pollution)`. This clearly shows the call with arguments being passed.

- **PPR List Segment 1 — data stored in the list:**
  Screenshot the `celestial_objects = [...]` declaration, showing at least 2 full dictionary entries. This proves data is being stored in the list.

- **PPR List Segment 2 — list being used:**
  Screenshot the `for obj in celestial_objects` loop in `main()`, including the `.append()` line and the sort. This shows the list is actively used to manage program complexity.

- **Written Response Q1 (input/output):** Lead with the four specific inputs, then explain the exact scoring pathway each triggers. Be precise about which variable controls which branch.

- **Written Response Q2a (iteration — zero executions):** State clearly "no such values exist" and explain *why*: the `best_months` list is always non-empty.

- **Written Response Q2b (two calls, same result):** Pre-compute exact scores for two different calls. Show the arithmetic. CollegeBoard rewards explicit reasoning over vague claims.

- **Written Response Q2c (procedure advantage):** Name the specific advantage as **abstraction / modularity** — and give a concrete scenario where it helps (e.g., changing the scoring formula only requires editing one function).

- **General tip:** Never say "my code" generically. Always reference specific line-level elements: function names, variable names, list names, exact parameter names.

## Wrap-Up

Between the ~92.9% MCQ score and a self-assessed 6/6 on the FRQ Create Task, this practice run shows the major exam-day risks for me are concentrated in a small set of MCQ patterns: abstraction wording, REPEAT UNTIL termination bugs, robot-grid orientation tracking, and graph fault-tolerance enumeration. The StarMatch project itself already lines up cleanly with every Create Task rubric row — what's left is rehearsing the exact PPR screenshots and written-response phrasings above so I don't lose easy points to vague language on test day.